# NB_patient_360

Builds `<catalog>.gold.patient_360` per the STTM *Silver to Gold (CareSync Health Network)*.

* **Grain:** one row per patient
* **Driving table:** `<catalog>.silver.patients`
* **Joins (LEFT):** `silver.hospitals` on `registered_hospital_id = hospital_id`, `silver.insurance_providers` on `insurance_provider_id`, aggregated `silver.lab_results` by `patient_id`
* **Filters:** none (all patients)
* **Load type:** FULL - `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
dbutils.widgets.text("catalog", "healthcare_databricks", "Catalog")
catalog = dbutils.widgets.get("catalog")

target_table = f"{catalog}.gold.patient_360"
print(f"Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per patient with hospital, insurance and lab-result summary (STTM silver to gold)'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                                AS total_tests,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count,
    MAX(test_date)                                          AS last_test_date
  FROM {catalog}.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  CAST(p.first_name AS STRING)                                           AS first_name,
  CAST(p.last_name AS STRING)                                            AS last_name,
  CAST(p.date_of_birth AS DATE)                                          AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  CAST(p.gender AS STRING)                                               AS gender,
  CAST(p.city AS STRING)                                                 AS city,
  CAST(p.state AS STRING)                                                AS state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  CAST(h.hospital_name AS STRING)                                        AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  CAST(ip.provider_name AS STRING)                                       AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                        AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                        AS critical_test_count,
  CAST(l.last_test_date AS DATE)                                         AS last_test_date
FROM {catalog}.silver.patients p
LEFT JOIN {catalog}.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {catalog}.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id
""")

In [0]:
# Grain check: gold row count must equal silver patients count and patient_id must be unique
display(spark.sql(f"""
SELECT
  (SELECT COUNT(*) FROM {catalog}.silver.patients)        AS silver_patients,
  COUNT(*)                                                AS gold_rows,
  COUNT(DISTINCT patient_id)                              AS distinct_patients,
  SUM(total_tests)                                        AS sum_total_tests,
  (SELECT COUNT(*) FROM {catalog}.silver.lab_results)     AS silver_lab_results,
  SUM(CASE WHEN hospital_name IS NULL THEN 1 ELSE 0 END)  AS missing_hospital_name,
  SUM(CASE WHEN insurance_provider_name IS NULL THEN 1 ELSE 0 END) AS missing_insurance_name
FROM {target_table}
"""))
display(spark.table(target_table).limit(10))

In [0]:
# Return the gold table record count to the caller (e.g. a job or dbutils.notebook.run)
updated_df = spark.table(target_table)

record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))